In [1]:
"""
LoanMate - 당신의 금융 계산 파트너
자연어 질의를 수학식으로 변환하여 금융 계산을 수행하는 애플리케이션
"""

import os
import streamlit as st
import plotly.graph_objects as go
from typing import Dict, List
from langchain_naver import ChatClovaX
from langchain.prompts import PromptTemplate
from langchain.schema import HumanMessage



class FinancialCalculator:
    """금융 계산 클래스"""
    
    @staticmethod
    def compound_interest(principal, rate, time):
        """복리 계산: A = P(1 + r)^t"""
        return principal * (1 + rate) ** time
    
    @staticmethod
    def simple_interest(principal, rate, time):
        """단리 계산: A = P(1 + rt)"""
        return principal * (1 + rate * time)
    
    @staticmethod
    def equal_payment_loan(principal, rate, time_years):
        """원리금균등상환 월납부액"""
        if rate == 0:
            return principal / (time_years * 12)
        
        monthly_rate = rate / 12
        n_payments = time_years * 12
        
        payment = principal * (monthly_rate * (1 + monthly_rate) ** n_payments) / \
                 ((1 + monthly_rate) ** n_payments - 1)
        return payment
    
    @staticmethod
    def bullet_payment_interest(principal, rate, time_years):
        """만기일시상환 총이자"""
        return principal * rate * time_years
    
    @staticmethod
    def max_loan_amount(monthly_payment, rate, time_years):
        """월납부액으로 가능한 최대 대출금액"""
        if rate == 0:
            return monthly_payment * time_years * 12
        
        monthly_rate = rate / 12
        n_payments = time_years * 12
        
        principal = monthly_payment * ((1 + monthly_rate) ** n_payments - 1) / \
                   (monthly_rate * (1 + monthly_rate) ** n_payments)
        return principal


class FinancialQueryProcessor:
    """자연어 금융 질의 처리 클래스"""
    
    def __init__(self):
        # Clova X 모델 초기화
        self.llm = ChatClovaX(
            api_key=os.getenv('NCP_CLOVASTUDIO_API_KEY'),
            model="HCX-005",
            temperature=0.2,
            max_tokens=2000,
        )
        
        self.prompt_template = PromptTemplate(
            input_variables=["query"],
            template="""
당신은 금융 계산 전문가입니다. 사용자의 질의를 분석하여 정확한 금융 계산을 수행하고 결과를 설명해주세요.

다음 공식들을 사용하여 계산하세요:

1. 복리 계산: A = P(1 + r)^t
   - A: 최종 금액, P: 원금, r: 연이자율, t: 기간(년)

2. 원리금균등상환 월납부액: PMT = P × [r(1+r)^n] / [(1+r)^n - 1]
   - PMT: 월납부액, P: 대출원금, r: 월이자율, n: 총 납부횟수

3. 만기일시상환 이자: I = P × r × t
   - I: 총 이자, P: 원금, r: 연이자율, t: 기간

4. 최대 대출액: P = PMT × [(1+r)^n - 1] / [r(1+r)^n]

사용자 질의: {query}

계산 과정과 결과를 명확하게 설명해주세요. 
단위 변환 (억=100,000,000원, 만=10,000원)도 적용하세요.
            """
        )
    
    def process_query(self, query: str) -> str:
        """자연어 질의 처리"""
        try:
            prompt = self.prompt_template.format(query=query)
            response = self.llm.invoke([HumanMessage(content=prompt)])
            
            # 텍스트 응답 그대로 반환
            return response.content
            
        except Exception as e:
            st.error(f"질의 처리 중 오류 발생: {e}")
            return f"죄송합니다. 처리 중 오류가 발생했습니다: {e}"


def create_comparison_chart(results: List[Dict]) -> go.Figure:
    """비교 차트 생성"""
    fig = go.Figure()
    
    periods = [r['period'] for r in results]
    amounts = [r['total_amount'] for r in results]
    interests = [r['total_interest'] for r in results]
    
    fig.add_trace(go.Bar(
        name='총 상환액',
        x=periods,
        y=amounts,
        marker_color='lightblue'
    ))
    
    fig.add_trace(go.Bar(
        name='총 이자',
        x=periods,
        y=interests,
        marker_color='coral'
    ))
    
    fig.update_layout(
        title='대출 기간별 비교',
        xaxis_title='기간',
        yaxis_title='금액 (원)',
        barmode='group',
        height=400
    )
    
    return fig


In [2]:
import pprint


loanmate = FinancialQueryProcessor()
query = "1000만원을 5년 동안 연 3%로 대출받을 때, 월납부액과 총 상환액을 계산해주세요."
pprint(loanmate.process_query(query))

ValidationError: 1 validation error for ChatClovaX
api_key
  Input should be a valid string [type=string_type, input_value=None, input_type=NoneType]
    For further information visit https://errors.pydantic.dev/2.11/v/string_type


**지원하는 계산 유형:**
- ✅ 복리/단리 이자 계산
- ✅ 대출 월납부액 계산
- ✅ 최대 대출 가능액 계산
- ✅ 여러 조건 비교 분석

**입력 예시:**
- "1억원을 연 4%로 3년간 예금하면?"
- "월 50만원씩 갚을 때 최대 대출액은?"
- "3년 vs 5년 대출 이자 비교해줘"



In [8]:
# 환경변수 - API 키 설정
# .env 파일에 NCP_CLOVASTUDIO_API_KEY 설정 필요
import dotenv
dotenv.load_dotenv()


# test function for FinancialCalculator

def test_financial_calculator():
    """금융 계산기 테스트"""
    calc = FinancialCalculator()
    
    print("=== LoanMate 금융 계산기 테스트 ===\n")
    
    # 1. 복리 계산 테스트
    print("1. 복리 계산 테스트")
    principal = 100000000  # 1억원
    rate = 0.04  # 연 4%
    time = 3  # 3년
    
    result = calc.compound_interest(principal, rate, time)
    interest = result - principal
    
    print(f"   원금: {principal:,}원")
    print(f"   이자율: {rate:.1%}")
    print(f"   기간: {time}년")
    print(f"   최종금액: {result:,.0f}원")
    print(f"   이자: {interest:,.0f}원")
    print(f"   공식: A = P(1+r)^t = {principal:,} × (1+{rate})^{time} = {result:,.0f}")
    
    # 2. 원리금균등상환 테스트
    print("\n2. 원리금균등상환 월납부액 계산")
    loan_amount = 100000000  # 1억원
    rate = 0.05  # 연 5%
    time = 3  # 3년
    
    monthly_payment = calc.equal_payment_loan(loan_amount, rate, time)
    total_payment = monthly_payment * time * 12
    total_interest = total_payment - loan_amount
    
    print(f"   대출금액: {loan_amount:,}원")
    print(f"   이자율: {rate:.1%}")
    print(f"   기간: {time}년")
    print(f"   월납부액: {monthly_payment:,.0f}원")
    print(f"   총상환액: {total_payment:,.0f}원")
    print(f"   총이자: {total_interest:,.0f}원")
    
    # 3. 만기일시상환 테스트
    print("\n3. 만기일시상환 이자 계산")
    principal = 100000000  # 1억원
    rate = 0.05  # 연 5%
    time = 3  # 3년
    
    total_interest = calc.bullet_payment_interest(principal, rate, time)
    total_amount = principal + total_interest
    
    print(f"   대출금액: {principal:,}원")
    print(f"   이자율: {rate:.1%}")
    print(f"   기간: {time}년")
    print(f"   총이자: {total_interest:,.0f}원")
    print(f"   만기상환액: {total_amount:,.0f}원")
    
    # 4. 최대 대출액 계산
    print("\n4. 최대 대출 가능액 계산")
    monthly_payment = 500000  # 월 50만원
    rate = 0.05  # 연 5%
    time = 3  # 3년
    
    max_loan = calc.max_loan_amount(monthly_payment, rate, time)
    
    print(f"   월납부액: {monthly_payment:,}원")
    print(f"   이자율: {rate:.1%}")
    print(f"   기간: {time}년")
    print(f"   최대대출액: {max_loan:,.0f}원")
    
    print("\n=== 테스트 완료 ===")

if __name__ == "__main__":
    test_financial_calculator()

=== LoanMate 금융 계산기 테스트 ===

1. 복리 계산 테스트
   원금: 100,000,000원
   이자율: 4.0%
   기간: 3년
   최종금액: 112,486,400원
   이자: 12,486,400원
   공식: A = P(1+r)^t = 100,000,000 × (1+0.04)^3 = 112,486,400

2. 원리금균등상환 월납부액 계산
   대출금액: 100,000,000원
   이자율: 5.0%
   기간: 3년
   월납부액: 2,997,090원
   총상환액: 107,895,230원
   총이자: 7,895,230원

3. 만기일시상환 이자 계산
   대출금액: 100,000,000원
   이자율: 5.0%
   기간: 3년
   총이자: 15,000,000원
   만기상환액: 115,000,000원

4. 최대 대출 가능액 계산
   월납부액: 500,000원
   이자율: 5.0%
   기간: 3년
   최대대출액: 16,682,851원

=== 테스트 완료 ===
